In [ ]:
import numpy as np

class Network:
    def __init__(self, input_dim, hidden_dim, output_dim):
        # Initialize W1, W2, W3
        # Initialize alpha (learnable scale for hidden_dim)
        self.W1 = np.random.randn(input_dim, hidden_dim) / np.sqrt(input_dim)
        self.W2 = np.random.randn(hidden_dim, hidden_dim) / np.sqrt(hidden_dim)
        self.W3 = np.random.randn(hidden_dim, output_dim) / np.sqrt(hidden_dim)
        self.alpha = np.random.randn(hidden_dim)

    def forward(self, X, Y_target):
        """
        Forward pass:
        1. Z1 = X @ W1
        2. H1 = Z1^2 + Z1
        3. H1_norm = H1 / max(abs(H1), axis=1, keepdims=True)

        4. Z2 = H1_norm @ W2
        5. H2 = Z2 * (2 - Z2)
        6. H2_scaled = H2 * alpha
        7. H2_mean = mean(H2_scaled, axis=0)
        8. H3 = H2_scaled + H2_mean
        9. Z3 = H3 + H1_norm

        10. logits = Z3 @ W3
        11. Apply softmax with stability
        12. Compute cross-entropy loss for integer labels

        Return scalar loss
        """
        _dict = {}
        _dict['x'] = X
        _dict['y'] = Y_target
        _dict['b'] = X.shape[0]

        out = X @ self.W1
        _dict['act6'] = out
        out = out ** 2 + out

        _dict['act5'] = out
        out_abs = np.abs(out)
        _dict['act4'] = out
        print(f"{out.shape}")
        out_max = np.max(out_abs, axis=1, keepdims=True)
        print(f"{out.shape}")
        out_norm = out / out_max

        _dict['out_norm'] = out_norm
        out = out_norm @ self.W2
        _dict['act3'] = out
        out = out * (2 - out)
        _dict['act2'] = out
        out = out * self.alpha
        _dict['act1'] = out
        out_mean = np.mean(out, axis=0)
        out = out + out_mean
        out = out + out_norm

        _dict['act0'] = out
        out = out @ self.W3

        out_exp = np.exp(out)
        p = out_exp / np.sum(out_exp, axis=1, keepdims=True)
        _dict['p'] = p
        assert np.allclose(p.sum(axis=1), 1)
        b = Y_target.shape[0]
        _loss = p[np.arange(b), Y_target]
        self.acts = _dict
        return -np.mean(np.log(_loss))

    def backward(self):
        """
        Compute gradients for W1, W2, W3, and alpha.
        Return dictionary with dW1, dW2, dW3, dalpha

        """
        acts = self.acts
        p, y_, b = self.acts['p'], self.acts['y'], self.acts['b']
        y = np.zeros_like(p)
        y[np.arange(b), y_] = 1
        grad = (p - y) / b

        dw3 = acts['act0'].T @ grad
        grad_z3 = grad @ self.W3.T # Z3

        grad_h2_mean = grad_z3.sum(axis=0)
        grad_h2_mean = np.ones_like(acts['act1']) * grad_h2_mean[np.newaxis, :] / acts['act1'].shape[0]
        grad_h2_scaled = grad_z3 + grad_h2_mean

        dalpha = np.sum(grad_h2_scaled * acts['act2'], axis=0)
        grad = grad_h2_scaled * self.alpha

        # f(x) * g(x) = f'(x) g(x) + f(x) g'(x)
        # Z2 * (2 - Z2)
        grad_h2 = grad * ( (2 - acts['act3']) *  (acts['act3'] * -acts['act3']) )
        dw2 = acts['out_norm'].T @ grad_h2
        grad_h1_norm = grad_h2 @ self.W2.T

        grad_h1_norm = grad_z3 + grad_h1_norm

        # h1 -> abs -> max
        _max = np.max(acts['act4'], axis=1)
        _max = _max[:, np.newaxis]
        grad_res = np.where(_max, grad_h1_norm, 0)

        grad_res = np.sign(acts['act4']) * grad_res
        grad_h1 = grad_res + grad_h1_norm

        # Z1^2 + Z1
        z1 = acts['act6']
        grad_h1 = grad_h1 * (2 * z1 + 1)

        dw1 = acts['x'].T @ grad_h1
        print(f"{grad_h1.shape=}")

        return {'dW1': dw1, 'dW2': dw2, 'dW3': dw3, 'dalpha': dalpha}


def test():
    net = Network(20, 30, 10)
    batch_size = 16

    X = np.random.randn(batch_size, 20) * 0.5
    Y = np.random.randint(0, 10, batch_size)

    loss = net.forward(X, Y)
    #print(f"{loss=}")
    grads = net.backward()

    print(f"Loss: {loss:.4f}")
    for name, grad in grads.items():
        print(f"{name}: {np.mean(np.abs(grad)):.6f}")

test()

(16, 30)
(16, 30)
grad_h1.shape=(16, 30)
Loss: 3.0615
dW1: 0.078107
dW2: 0.004001
dW3: 0.055437
dalpha: 0.038969


In [ ]:
import numpy as np

class SplitNetwork:
    def __init__(self, input_dim, hidden_dim, output_dim):
        # Initialize W1, W2
        # Initialize gamma (learnable parameter for hidden_dim)
        self.W1 = np.random.randn(input_dim, hidden_dim) / np.sqrt(input_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) / np.sqrt(output_dim)
        self.gamma = np.random.randn(hidden_dim)

    def forward(self, X, Y_target):
        """
        Forward pass:
        1. Z1 = X @ W1
        2. H1 = relu(Z1)
        3. H1_sum = sum(H1, axis=0)
        4. H1_gated = H1 * gamma
        5. H2 = H1_gated + H1_sum
        6. H3 = H2 + H1
        7. logits = H3 @ W2
        8. Apply softmax with stability
        9. Compute cross-entropy loss for integer labels

        Return scalar loss
        """
        acts = {}
        acts['x'] = X
        acts['y'] = Y_target
        acts['b'] = X.shape[0]

        z1 = X @ self.W1
        acts['z1'] = z1
        h1 = (z1 > 0) * z1
        acts['h1'] = h1
        h1_sum = np.sum(h1, axis=0)
        acts['h1_sum'] = h1_sum

        h1_gated = h1 * self.gamma
        acts['h1_gated'] = h1_gated
        h2 = h1_gated + h1_sum
        acts['h2'] = h2
        h3 = h2 + h1
        acts['h3'] = h3

        logits = h3 @ self.W2

        logits = logits - logits.max(axis=1, keepdims=True)
        logits_exp = np.exp(logits)
        p = logits_exp / np.sum(logits_exp, axis=1, keepdims=True)
        assert np.allclose(p.sum(axis=1), 1)
        acts['p'] = p
        self.acts = acts
        return -np.mean(np.log(p[np.arange(acts['b']), Y_target]))


    def backward(self):
        """
        Compute gradients for W1, W2, and gamma.
        Return dictionary with dW1, dW2, dgamma.

          x -> W1 -> Z1 -> ReLu() -> H1 -> sum(axis=0) -> H1_sum ->  H2 -> H3 -> W2 -> logits
                                   |                            |         |
                                    ---> gamma --> H1_gated ---->         |
                                    ------------------------------------ --

        """
        acts = self.acts
        y_, p, b = acts['y'], acts['p'], acts['b']
        y = np.zeros_like(p)
        y[np.arange(b), y_] = 1

        # CE
        grad_logits = (p - y) / b
        print(f"{grad_logits.shape=}")

        dw2 = acts['h3'].T @ grad_logits
        grad_h3 = grad_logits @ self.W2.T

        grad_h2 = grad_h3
        grad_h1 = grad_h3

        grad_h1_gated = grad_h2
        grad_h1_sum = grad_h2

        # grad_h1_sum
        # [B, H] -> H -> [B, H]
        h1 = acts['h1']
        grad_h1_sum = np.sum(grad_h1_sum, axis=0) # [H]
        grad_h1_sum = grad_h1_sum[np.newaxis, :] * np.ones_like(h1) # [B, H]

        dgamma = np.sum(grad_h1_gated * h1, axis=0)
        grad_h1_gated = grad_h1_gated * self.gamma

        grad_h1 = grad_h1 + grad_h1_sum + grad_h1_gated

        grad_h1 = grad_h1 * (acts['z1'] > 0)


        print(f"{dgamma.shape=}")
        print(f"{grad_h1_gated.shape=}")
        print(f"{grad_h1_sum.shape=}")
        print(f"{grad_h1.shape=}")
        #grad_h1 = grad_h1 + ...

        dw1 = acts['x'].T @ grad_h1

        print(f"{grad_h3.shape=}")

        return {'dw1': dw1, 'dw2': dw2, 'dgamma': dgamma}

def test():
    net = SplitNetwork(25, 20, 10)
    batch_size = 8

    X = np.random.randn(batch_size, 25) * 0.5
    Y = np.random.randint(0, 10, batch_size)

    loss = net.forward(X, Y)
    print(f"{loss=}")
    grads = net.backward()

    print(f"Loss: {loss:.4f}")
    for name, grad in grads.items():
        print(f"{name}: {np.mean(np.abs(grad)):.6f}")

test()

loss=np.float64(2.6597142880897575)
grad_logits.shape=(8, 10)
dgamma.shape=(20,)
grad_h1_gated.shape=(8, 20)
grad_h1_sum.shape=(8, 20)
grad_h1.shape=(8, 20)
grad_h3.shape=(8, 20)
Loss: 2.6597
dw1: 0.122035
dw2: 0.217961
dgamma: 0.033564


In [ ]:
# Fix Backward Pass

In [ ]:
import numpy as np

class DebugNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.01
        self.beta = np.zeros(hidden_dim)

    def forward(self, X, Y_target):
        """
        Forward pass (THIS IS CORRECT):
        1. Z1 = X @ W1
        2. H1 = sigmoid(Z1)
        3. H1_shifted = H1 + beta
        4. H1_mean = mean(H1_shifted, axis=1, keepdims=True)
        5. H2 = H1_shifted - H1_mean
        6. out = H2 @ W2
        7. loss = mean((out - Y_target)^2)
        """
        self.acts = {}
        self.acts['X'] = X
        self.acts['Y'] = Y_target

        Z1 = X @ self.W1
        self.acts['Z1'] = Z1

        H1 = 1 / (1 + np.exp(-Z1))
        self.acts['H1'] = H1

        H1_shifted = H1 + self.beta
        self.acts['H1_shifted'] = H1_shifted

        H1_mean = np.mean(H1_shifted, axis=1, keepdims=True)
        self.acts['H1_mean'] = H1_mean

        H2 = H1_shifted - H1_mean
        self.acts['H2'] = H2

        out = H2 @ self.W2
        self.acts['out'] = out

        loss = np.mean((out - Y_target)**2)
        return loss

    def backward(self):
        """
        BACKWARD PASS HAS BUGS - FIX THEM
        """
        batch = self.acts['X'].shape[0]

        # Loss gradient
        grad = 2 * (self.acts['out'] - self.acts['Y']) / self.acts['out'].size
        print(f"{grad.shape=}")
        #print(f"{grad=}")

        # Through W2
        self.dW2 = self.acts['H2'].T @ grad
        grad = grad @ self.W2.T

        # Through H2 = H1_shifted - H1_mean
        grad_H1_shifted = grad
        grad_H1_mean = -grad
        # looks ok.

        # Through mean
        # 4. H1_mean = mean(H1_shifted, axis=1, keepdims=True)
        # [B,D] -> [B] -> [B,D]
        print(f"{grad_H1_mean.shape}")
        grad_H1_mean = np.sum(grad_H1_mean, axis=1) # [B]

        grad_H1_mean = grad_H1_mean[:, np.newaxis] * np.ones_like(self.acts['H1_shifted']) / self.acts['H1_shifted'].shape[1]
        #print(f"{grad_H1_mean.shape=}")
        grad_H1_shifted = grad_H1_mean + grad_H1_shifted
        print(f"{grad_H1_shifted.shape=}")

        # Through H1_shifted = H1 + beta
        grad_H1 = grad_H1_shifted
        self.dbeta = np.sum(grad_H1_shifted, axis=0)

        # Through sigmoid
        sig = self.acts['H1']
        grad = grad_H1 * sig * (1 - sig)

        # Through W1
        self.dW1 = self.acts['X'].T @ grad

        return {'dW1': self.dW1, 'dW2': self.dW2, 'dbeta': self.dbeta}

def test():
    np.random.seed(42)
    net = DebugNet(10, 20, 5)

    X = np.random.randn(8, 10) * 0.5
    Y = np.random.randn(8, 5) * 0.5

    loss = net.forward(X, Y)
    grads = net.backward()

    print(f"Loss: {loss:.4f}")
    print(f"dW1 shape: {grads['dW1'].shape}, mean: {np.mean(np.abs(grads['dW1'])):.6f}")
    print(f"dW2 shape: {grads['dW2'].shape}, mean: {np.mean(np.abs(grads['dW2'])):.6f}")
    print(f"dbeta shape: {grads['dbeta'].shape}, mean: {np.mean(np.abs(grads['dbeta'])):.6f}")

test()

grad.shape=(8, 5)
(8, 20)
grad_H1_shifted.shape=(8, 20)
Loss: 0.2535
dW1 shape: (10, 20), mean: 0.000117
dW2 shape: (20, 5), mean: 0.000162
dbeta shape: (20,), mean: 0.001099


In [ ]:
# Fix Code
import numpy as np

class BuggyNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.W2 = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.W3 = np.random.randn(hidden_dim, output_dim) * 0.01
        self.alpha = np.ones(hidden_dim) * 0.1

    def forward(self, X, Y_target):
        """
        Forward pass (THIS IS CORRECT):
        1. Z1 = X @ W1
        2. H1 = relu(Z1)
        3. H1_scaled = H1 * alpha
        4. Z2 = H1_scaled @ W2
        5. H2 = sigmoid(Z2)
        6. H2_max = max(H2, axis=0)
        7. H3 = H2 + H2_max + H1_scaled
        8. out = H3 @ W3
        9. loss = mean((out - Y_target)^2)
        """
        self.acts = {}
        self.acts['X'] = X
        self.acts['Y'] = Y_target

        Z1 = X @ self.W1
        self.acts['Z1'] = Z1

        H1 = Z1 * (Z1 > 0)
        self.acts['H1'] = H1

        H1_scaled = H1 * self.alpha
        self.acts['H1_scaled'] = H1_scaled

        Z2 = H1_scaled @ self.W2
        self.acts['Z2'] = Z2

        H2 = 1 / (1 + np.exp(-Z2))
        self.acts['H2'] = H2

        H2_max = np.max(H2, axis=0)
        self.acts['H2_max'] = H2_max

        H3 = H2 + H2_max + H1_scaled
        self.acts['H3'] = H3

        out = H3 @ self.W3
        self.acts['out'] = out

        loss = np.mean((out - Y_target)**2)
        return loss

    def backward(self):
        """
        BACKWARD PASS HAS BUGS - FIX THEM
        """
        batch = self.acts['X'].shape[0]

        # Loss gradient
        grad = 2 * (self.acts['out'] - self.acts['Y']) / self.acts['Y'].size

        # Through W3
        self.dW3 = self.acts['H3'].T @ grad
        grad_H3 = grad @ self.W3.T

        # Through H3 = H2 + H2_max + H1_scaled
        grad_H2 = grad_H3
        grad_H2_max = grad_H3
        grad_H1_scaled = grad_H3

        # Through max (H2_max = max(H2, axis=0))
        # H2 -> H2_max -> broadcast
        # [B, D] -> [D] -> [B, D]
        max_mask = (self.acts['H2'] == self.acts['H2_max'])
        grad_H2_max = np.sum(grad_H2_max, axis=0)
        print(f"{grad_H2_max.shape=}")
        grad_H2 = grad_H2_max * max_mask
        print(f"{grad_H2.shape=}")

        # Through sigmoid
        sig = self.acts['H2']
        grad_Z2 = grad_H2 * sig * (1 - sig)

        # Through W2
        self.dW2 = self.acts['H1_scaled'].T @ grad_Z2
        grad_H1_scaled = grad_H1_scaled + grad_Z2 @ self.W2.T

        # Through H1_scaled = H1 * alpha
        #print(f"{grad_H1_scaled.shape=}, {self.acts['H1'].shape=}")
        self.dalpha = np.sum(grad_H1_scaled * self.acts['H1'], axis=0)
        grad_H1 = grad_H1_scaled * self.alpha

        # Through relu
        grad_Z1 = grad_H1 * (self.acts['Z1'] > 0)

        # Through W1
        self.dW1 = self.acts['X'].T @ grad_Z1

        return {'dW1': self.dW1, 'dW2': self.dW2, 'dW3': self.dW3, 'dalpha': self.dalpha}

def test():
    np.random.seed(42)
    net = BuggyNet(15, 25, 10)

    X = np.random.randn(12, 15) * 0.5
    Y = np.random.randn(12, 10) * 0.5

    loss = net.forward(X, Y)
    grads = net.backward()

    print(f"Loss: {loss:.4f}")
    print(f"dW1 shape: {grads['dW1'].shape}, mean: {np.mean(np.abs(grads['dW1'])):.6f}")
    print(f"dW2 shape: {grads['dW2'].shape}, mean: {np.mean(np.abs(grads['dW2'])):.6f}")
    print(f"dW3 shape: {grads['dW3'].shape}, mean: {np.mean(np.abs(grads['dW3'])):.6f}")
    print(f"dalpha shape: {grads['dalpha'].shape}, mean: {np.mean(np.abs(grads['dalpha'])):.6f}")

test()

grad_H2_max.shape=(25,)
grad_H2.shape=(12, 25)
Loss: 0.2729
dW1 shape: (15, 25), mean: 0.000029
dW2 shape: (25, 25), mean: 0.000000
dW3 shape: (25, 10), mean: 0.026471
dalpha shape: (25,), mean: 0.000008


In [ ]:
import numpy as np

class MaxNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.01

    def forward(self, X, Y_target):
        """
        Forward pass:
        1. H1 = X @ W1
        2. H1_relu = relu(H1)
        3. H1_max = max(H1_relu, axis=0)
        4. H2 = H1_relu + H1_max

        5. H2_max_pool = max(H2, axis=1)
        6. H3 = H2 - H2_max_pool

        7. out = H3 @ W2
        8. loss = mean((out - Y_target)^2)
        """
        acts = {}
        acts['x'], acts['y'] = X, Y_target

        h1 = X @ self.W1
        acts['h1'] = h1
        h1_relu = h1 * (h1 > 0)
        acts['h1_relu'] = h1_relu
        h1_max = np.max(h1_relu, axis=0, keepdims=True)
        acts['h1_max'] = h1_max

        h2 = h1_relu + h1_max
        acts['h2'] = h2

        h2_max = np.max(h2, axis=1, keepdims=True)
        acts['h2_max'] = h2_max

        h3 = h2 - h2_max
        acts['h3'] = h3

        logits = h3 @ self.W2

        logits = logits - logits.max(axis=1, keepdims=True)
        logits_exp = np.exp(logits)
        p = logits_exp / logits_exp.sum(axis=1, keepdims=True)
        acts['p'] = p
        assert np.allclose(p.sum(axis=1), 1)
        print(f"{p.shape=},  {acts['y'].shape=}")
        self.acts = acts
        return np.mean(p - acts['y']) ** 2

    def backward(self):
        """
        Compute gradients for W1 and W2.
        Focus on correctly handling both max operations.
        """
        acts = self.acts
        grad = 2 * (acts['p'] - acts['y']) / self.acts['p'].size


        grad = grad @ self.W2.T

        grad_h2 = grad
        grad_h2_max = -grad

        # h2_max = np.max(h2, axis=1, keepdims=True)
        # [B, D] -> [B] -> [B,D]
        grad_h2_max = np.sum(grad_h2_max, axis=1)
        mask = (acts['h2'] == acts['h2_max'])
        grad_h2_max = grad_h2_max[:, np.newaxis] * mask


        grad_h2 = grad_h2 + grad_h2_max


        grad_h1_relu = grad_h2
        grad_h1_max = grad_h2

        # h1_max = np.max(h1_relu, axis=0, keepdims=True)
        # [B, D] -> [D] -> [B,D]
        grad_h1_max = np.sum(grad_h1_max, axis=0)
        mask = (acts['h1_relu'] == acts['h1_max'])
        grad_h1_max = grad_h1_max[np.newaxis, :] * mask

        grad_h1_relu = grad_h1_relu + grad_h1_max
        grad_h1_relu = grad_h1_relu * (acts['h1'] > 0)

        grad_h1_relu @ self.W1.T



def test():
    np.random.seed(42)
    net = MaxNet(10, 15, 5)

    X = np.random.randn(8, 10) * 0.5
    Y = np.random.randn(8, 5) * 0.5

    loss = net.forward(X, Y)
    grads = net.backward()

    # print(f"Loss: {loss:.4f}")
    # print(f"dW1 mean: {np.mean(np.abs(grads['dW1'])):.6f}")
    # print(f"dW2 mean: {np.mean(np.abs(grads['dW2'])):.6f}")

test()

p.shape=(8, 5),  acts['y'].shape=(8, 5)
grad_h1_max.shape=(8, 15)


In [ ]:
import numpy as np

class WarmupNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.01
        self.gamma = np.ones(hidden_dim) * 0.5

    def forward(self, X, Y_target):
        """
        Forward pass:
        1. Z1 = X @ W1

        2. H1 = Z1^2
        3. H1_mean = mean(H1, axis=0)
        4. H1_centered = H1 - H1_mean
        5. H2 = relu(H1_centered)
        6. H2_max = max(H2, axis=1, keepdims=True)
        7. H3 = H2 / (H2_max + 1e-8)

        8. H3_scaled = H3 * gamma
        9. out = H3_scaled @ W2
        10. loss = mean((out - Y_target)^2)

        """
        acts = {}
        acts['x'] = X
        acts['y'] = Y_target

        z1 = X @ self.W1
        acts['z1'] = z1
        h1 = z1 ** 2
        acts['h1'] = h1
        h1_mean = np.mean(h1, axis=0) # [B, D] -> [D]
        acts['h1_mean'] = h1_mean
        h1_centered = h1 - h1_mean
        acts['h1_centered'] = h1_centered

        h2 = h1_centered * (h1_centered > 0) # relu
        acts['h2'] = h2
        h2_max = np.max(h2, axis=1, keepdims=True) # [B, D] -> [B]
        acts['h2_max'] = h2_max
        h3 = h2 / (h2_max  + 1e-8)
        acts['h3'] = h3
        h3_scaled = h3 * self.gamma
        acts['h3_scaled'] = h3_scaled
        out = h3_scaled @ self.W2
        acts['out'] = out
        loss = np.mean((out - acts['y'])**2)
        self.acts = acts
        return loss


    def backward(self):
        """
        Compute gradients for W1, W2, and gamma.
        Return dictionary with dW1, dW2, dgamma.

        branch out in H3 and H1_centered
        """
        acts = self.acts
        grad = 2 * (acts['out'] - acts['y']) / acts['y'].size

        grad_h3_scaled = grad @ self.W2.T
        dw2 = acts['h3_scaled'].T @ grad

        grad_h3 = grad_h3_scaled * self.gamma
        dgamma = np.sum(grad_h3_scaled * acts['h3'], axis=0)

        grad_h2 = grad_h3
        grad_h2_max = 1 / grad_h3

        # # [B,D] <- [B] <- [B, D]
        grad_h2_max = np.sum(grad_h2_max, axis=1)
        mask = (acts['h2'] == acts['h2_max'])
        grad_h2_max = grad_h2_max[:, np.newaxis] * mask

        grad_h2 = grad_h2 + grad_h2_max

        grad_h1_centered = (acts['h1_centered'] > 0) * grad_h2



        dh1 = grad_h1_centered
        d_h1_mean = - grad_h1_centered

        # # [B, D] <- [D] <- [B, D]
        d_h1_mean = np.sum(d_h1_mean, axis=0)
        d_h1_mean = np.ones_like(acts['h1']) * d_h1_mean[np.newaxis, :] / acts['x'].shape[0]

        dh1 = dh1 + d_h1_mean
        dz1 = dh1 * (2 * acts['z1'])


        print(f"{dz1.shape=}")






def test():
    np.random.seed(42)
    net = WarmupNet(12, 18, 6)

    X = np.random.randn(10, 12) * 0.5
    Y = np.random.randn(10, 6) * 0.5

    loss = net.forward(X, Y)
    print(f"{loss=}")
    grads = net.backward()

    # print(f"Loss: {loss:.4f}")
    # print(f"dW1 mean: {np.mean(np.abs(grads['dW1'])):.6f}")
    # print(f"dW2 mean: {np.mean(np.abs(grads['dW2'])):.6f}")
    # print(f"dgamma mean: {np.mean(np.abs(grads['dgamma'])):.6f}")

test()

loss=np.float64(0.2802212139084549)
dz1.shape=(10, 18)
